# TIDN — Manifold Geometry Demo\n\nVisualize the difference between Euclidean distance and Fisher-Rao geodesic distance.\nThis demonstrates why TIDN uses information-geometric distances instead of dot products.

In [ ]:
import torch\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom tidn.layers.geometry import (\n    fisher_rao_distance_gaussian,\n    ExpFamilyEmbedding,\n)\n\n%matplotlib inline

In [ ]:
# Create an embedding that maps 2D points to Gaussian distributions
emb = ExpFamilyEmbedding(in_dim=2, manifold_dim=2, min_scale=0.01)

# Sample a grid of 2D points
grid = torch.stack(torch.meshgrid(
    torch.linspace(-2, 2, 20),
    torch.linspace(-2, 2, 20),
    indexing='ij'
), dim=-1).reshape(-1, 2)

mu, sigma = emb(grid)

# Compute Euclidean distance matrix
euclidean_dist = torch.cdist(grid, grid)

# Compute Fisher-Rao distance matrix
# Note: signature is (mu1, sigma1, mu2, sigma2)
mu_i = mu.unsqueeze(0)
mu_j = mu.unsqueeze(1)
s_i = sigma.unsqueeze(0)
s_j = sigma.unsqueeze(1)
fisher_dist = fisher_rao_distance_gaussian(mu_i, s_i, mu_j, s_j)

print(f"Euclidean range: [{euclidean_dist.min():.3f}, {euclidean_dist.max():.3f}]")
print(f"Fisher-Rao range: [{fisher_dist.min():.3f}, {fisher_dist.max():.3f}]")

In [ ]:
# Compare distance matrices\nfig, axes = plt.subplots(1, 3, figsize=(15, 4))\n\naxes[0].imshow(euclidean_dist, cmap='viridis')\naxes[0].set_title('Euclidean Distance')\n\naxes[1].imshow(fisher_dist, cmap='viridis')\naxes[1].set_title('Fisher-Rao Distance')\n\naxes[2].imshow((fisher_dist - euclidean_dist).abs(), cmap='hot')\naxes[2].set_title('Absolute Difference')\n\nplt.tight_layout()\nplt.show()

## Key Observation\n\nThe Fisher-Rao distance is NOT a simple rescaling of Euclidean distance.\nPoints that are close in Euclidean space may be far apart on the statistical manifold\n(because their induced distributions are very different), and vice versa.\n\nThis is why dot-product attention in Transformers misses important structural\nrelationships that TIDN's resonance routing captures.